# Tall-crop bleed labeling

Fresh stratified visual-inspection pass over the tall-bucket crops (aspect
ratio w/h < 1.0) from the 300-crop OCR val set, replacing the prior 74-crop
review that was done in a past chat session and never saved to disk.

Runs entirely locally (no Drive/Colab mount needed) -- the 300-crop val set
and its images already live under `data/paddle_finetune/dataset/` in this
repo. Does NOT touch `notebooks/Labelling.ipynb`, `manual_labels.csv`, or
any file under `data/paddle_finetune/dataset/` or
`data/paddle_finetune/zero_shot_rotation_test/` -- this notebook only reads
those, and only writes to a new file,
`data/paddle_finetune/tall_crop_bleed_labels.csv`.

In [1]:
# ---- CELL 1: Config ----
import csv
import os
import random
from datetime import datetime

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATASET_DIR = os.path.join(REPO_ROOT, "data/paddle_finetune/dataset")
ROTATION_RESULTS_CSV = os.path.join(REPO_ROOT, "data/paddle_finetune/zero_shot_rotation_test/results.csv")
OUTPUT_CSV = os.path.join(REPO_ROOT, "data/paddle_finetune/tall_crop_bleed_labels.csv")

print("REPO_ROOT:", REPO_ROOT)
assert os.path.isfile(ROTATION_RESULTS_CSV), f"missing {ROTATION_RESULTS_CSV}"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

REPO_ROOT: /home/shashwatkumar/Testing/container_ocr_test


In [4]:
# ---- CELL 2: Load the tall bucket from the already-verified rotation-test
# results, and queue up whichever of those crops aren't labeled yet ----
#
# image_path + bucket come straight from zero_shot_rotation_test.py's own
# output (data/paddle_finetune/zero_shot_rotation_test/results.csv) -- the
# tall/wide split itself is NOT recomputed here, so this stays consistent
# with the 249/300 (83%) figure already reported from that script.
#
# aspect_ratio is not a column in that CSV (it only stores the resulting
# `bucket` label), so it's derived here directly from each image's actual
# pixel dimensions via PIL -- same w/h formula the rotation script used to
# assign the bucket in the first place, just not re-deciding the bucket.

from PIL import Image

def load_tall_bucket():
    with open(ROTATION_RESULTS_CSV, newline='') as f:
        rows = list(csv.DictReader(f))
    tall = [r["image_path"] for r in rows if r["bucket"] == "tall"]
    assert len(tall) == len(set(tall)), "duplicate image_path in tall bucket"
    return tall

def aspect_ratio_of(rel_path):
    with Image.open(os.path.join(DATASET_DIR, rel_path)) as im:
        w, h = im.size
    return w / h

def load_already_labeled():
    done = set()
    if os.path.exists(OUTPUT_CSV) and os.path.getsize(OUTPUT_CSV) > 0:
        with open(OUTPUT_CSV, newline='') as f:
            for row in csv.DictReader(f):
                done.add(row["image_path"])
    return done

tall_paths = load_tall_bucket()
already_done = load_already_labeled()
remaining = [p for p in tall_paths if p not in already_done]

random.shuffle(remaining)  # no fixed seed -- order only, not a sub-sample:
                           # every path in `remaining` is still queued, just
                           # reviewed in a fresh random order each session so
                           # a partial/interrupted pass is still a
                           # representative cross-section rather than
                           # biased toward filename order.
batch = remaining  # full tall population, per instructions (ask before shrinking)

print(f"Tall bucket (verified via rotation-test results.csv): {len(tall_paths)}")
print(f"Already labeled in {os.path.basename(OUTPUT_CSV)}:      {len(already_done)}")
print(f"Queued for this/future sessions:                        {len(remaining)}")

Tall bucket (verified via rotation-test results.csv): 249
Already labeled in tall_crop_bleed_labels.csv:      0
Queued for this/future sessions:                        249


In [5]:
# ---- CELL 3: Interactive bleed-labeling loop -- ipywidgets version ----
#
# Same mechanism as Labelling.ipynb Cell 5 (persistent Output + Button
# widgets, no blocking input() calls, incremental writes with an immediate
# flush after every row) -- adapted from free-text code entry to fixed
# bleed-category buttons, since this pass is classifying crop *quality*
# (clean vs. which kind of bleed), not transcribing the container code.
#
# Controls: click one of the four category buttons, or Skip. An optional
# free-text Notes box is read at click time and saved alongside whichever
# category was chosen, then cleared for the next crop.

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

FIELDNAMES = ["image_path", "bucket", "aspect_ratio", "bleed_label", "notes", "timestamp"]

state = {
    "index": 0,
    "labeled_this_session": 0,
    "csv_file": open(OUTPUT_CSV, "a", newline=''),
}
state["writer"] = csv.writer(state["csv_file"])
if os.path.getsize(OUTPUT_CSV) == 0:
    state["writer"].writerow(FIELDNAMES)
    state["csv_file"].flush()

image_out = widgets.Output()
status_out = widgets.Output()
notes_box = widgets.Text(placeholder="optional notes", layout=widgets.Layout(width="300px"))
btn_clean = widgets.Button(description="Clean single-column", button_style="success")
btn_right = widgets.Button(description="Right-column bleed", button_style="warning")
btn_below = widgets.Button(description="Below-column bleed", button_style="warning")
btn_other = widgets.Button(description="Other/tangled", button_style="danger")
btn_skip = widgets.Button(description="Skip")
btn_stop = widgets.Button(description="Stop", button_style="danger")

controls = widgets.HBox([notes_box, btn_clean, btn_right, btn_below, btn_other, btn_skip, btn_stop])
display(image_out, controls, status_out)


def show_current_crop():
    with image_out:
        clear_output(wait=True)
        if state["index"] >= len(batch):
            print(f"Batch finished. Labeled {state['labeled_this_session']} crops this session.")
            return
        rel_path = batch[state["index"]]
        img = Image.open(os.path.join(DATASET_DIR, rel_path)).convert("RGB")
        display(img)
        print(f"{rel_path}  —  {state['index'] + 1}/{len(batch)}")
    notes_box.value = ""


def write_row(bleed_label):
    rel_path = batch[state["index"]]
    ar = aspect_ratio_of(rel_path)
    state["writer"].writerow([rel_path, "tall", round(ar, 4), bleed_label,
                               notes_box.value.strip(), datetime.now().isoformat()])
    state["csv_file"].flush()


def advance():
    state["index"] += 1
    show_current_crop()


def make_handler(label):
    def _handler(_):
        write_row(label)
        state["labeled_this_session"] += 1
        advance()
    return _handler


def on_skip(_):
    write_row("skip")
    advance()


def on_stop(_):
    with status_out:
        clear_output(wait=True)
        print(f"Stopped. Labeled {state['labeled_this_session']} crops this session.")
    state["csv_file"].close()


btn_clean.on_click(make_handler("clean_single_column"))
btn_right.on_click(make_handler("right_column_bleed"))
btn_below.on_click(make_handler("below_column_bleed"))
btn_other.on_click(make_handler("other_tangled"))
btn_skip.on_click(on_skip)
btn_stop.on_click(on_stop)

show_current_crop()

Output()

Output()

In [ ]:
# ---- CELL 4: Sanity check after a session ----
import pandas as pd
df = pd.read_csv(OUTPUT_CSV)
print(f"Total rows in CSV: {len(df)}")
print(df["bleed_label"].value_counts())
print()
print(f"Tall-bucket coverage: {df['image_path'].nunique()} / {len(tall_paths)} labeled so far")

In [ ]:
# ---- CELL 5: Review / correct already-labeled tall crops ----
#
# Read-and-fix pass over tall_crop_bleed_labels.csv -- does NOT touch Cell 3
# or its open append-mode file handle, and never changes which image_paths
# exist in the CSV (only edits bleed_label/notes/timestamp on rows already
# there). Cell 2's `already_done` resume set for Cell 3 is computed from
# the set of image_paths in the CSV, which this cell never adds to or
# removes from -- so Cell 3's resumability is unaffected no matter how many
# times this cell runs, before or after it.
#
# Ground-truth source: data/paddle_finetune/dataset/val.txt directly (same
# VAL_TXT file scripts/paddle_finetune/zero_shot_rotation_test.py reads).
# That script's own results.csv also carries a `ground_truth` column, but
# it's a derived copy of val.txt, not the original -- read val.txt itself
# here rather than the copy.
#
# Concurrency note: only run this cell OR actively label in Cell 3 at a
# time, not both in the same instant. That said, "Save correction" below
# re-reads the whole CSV from disk immediately before writing (not off a
# stale in-memory snapshot from whenever this cell started), so it can
# never blow away rows Cell 3 appended earlier in the same session -- the
# only unsafe window is two writes landing in the exact same instant,
# which isn't possible from a single person clicking one button at a time.

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

VAL_TXT = os.path.join(DATASET_DIR, "val.txt")


def load_ground_truth():
    """Primary source: val.txt. Fallback: the `ground_truth` column already
    in ROTATION_RESULTS_CSV, for paths val.txt no longer contains.

    val.txt was regenerated (2026-07-14 13:24) AFTER
    zero_shot_rotation_test/results.csv was produced (2026-07-14 12:23) --
    confirmed 86 of the 249 tall-bucket paths (and 44 of the rows already in
    tall_crop_bleed_labels.csv) are no longer in the current val.txt at all,
    though their image files still exist on disk and aren't in train.txt
    either -- they were dropped from both splits in that later regen, not
    moved or deleted. So val.txt alone is NOT a complete ground-truth source
    for this review pass; falling back to the rotation results.csv copy for
    those paths and flagging it on-screen, rather than silently showing
    "not found" for over a third of already-labeled rows."""
    gt = {}
    with open(VAL_TXT, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            img_path, label = line.split("\t", 1) if "\t" in line else line.split(None, 1)
            gt[img_path.strip()] = label.strip()

    gt_fallback = {}
    if os.path.isfile(ROTATION_RESULTS_CSV):
        with open(ROTATION_RESULTS_CSV, newline='') as f:
            for row in csv.DictReader(f):
                if row["image_path"] not in gt:
                    gt_fallback[row["image_path"]] = row["ground_truth"]

    return gt, gt_fallback


ground_truth, ground_truth_fallback = load_ground_truth()


def lookup_ground_truth(rel_path):
    """Returns (label, source_tag) so the caller can flag stale lookups."""
    if rel_path in ground_truth:
        return ground_truth[rel_path], "val.txt"
    if rel_path in ground_truth_fallback:
        return ground_truth_fallback[rel_path], "STALE: rotation results.csv only, not in current val.txt"
    return None, "not found anywhere"


def read_all_labeled_rows():
    if not os.path.exists(OUTPUT_CSV) or os.path.getsize(OUTPUT_CSV) == 0:
        return [], []
    with open(OUTPUT_CSV, newline='') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)
    return fieldnames, rows


_fieldnames0, _rows0 = read_all_labeled_rows()
review_paths = [r["image_path"] for r in _rows0]  # stable browsing order for this session;
                                                    # rerun this cell to pick up rows Cell 3
                                                    # labels afterward

review_state = {"pos": 0, "pending_label": None}

image_out3 = widgets.Output()
status_out3 = widgets.Output()
notes_box3 = widgets.Text(layout=widgets.Layout(width="300px"))
btn_clean3 = widgets.Button(description="Clean single-column", button_style="success")
btn_right3 = widgets.Button(description="Right-column bleed", button_style="warning")
btn_below3 = widgets.Button(description="Below-column bleed", button_style="warning")
btn_other3 = widgets.Button(description="Other/tangled", button_style="danger")
btn_skip3 = widgets.Button(description="Skip")
btn_save3 = widgets.Button(description="Save correction", button_style="info")
btn_back3 = widgets.Button(description="< Back")
btn_next3 = widgets.Button(description="Next >")

controls3a = widgets.HBox([btn_back3, btn_next3])
controls3b = widgets.HBox([btn_clean3, btn_right3, btn_below3, btn_other3, btn_skip3])
controls3c = widgets.HBox([notes_box3, btn_save3])
display(image_out3, controls3a, controls3b, controls3c, status_out3)


def current_row():
    """Re-reads OUTPUT_CSV fresh every call, so this reflects the latest
    data even if Cell 3 appended new rows since this cell last ran."""
    _, rows = read_all_labeled_rows()
    by_path = {r["image_path"]: r for r in rows}
    rel_path = review_paths[review_state["pos"]]
    return by_path.get(rel_path)


def show_current():
    if not review_paths:
        with image_out3:
            clear_output(wait=True)
            print("No labeled rows yet in tall_crop_bleed_labels.csv -- run Cell 3 first.")
        return
    rel_path = review_paths[review_state["pos"]]
    row = current_row()
    with image_out3:
        clear_output(wait=True)
        img = Image.open(os.path.join(DATASET_DIR, rel_path)).convert("RGB")
        display(img)
        print(f"{rel_path}  \u2014  {review_state['pos'] + 1}/{len(review_paths)}")
        gt_label, gt_source = lookup_ground_truth(rel_path)
        print(f"ground_truth: {gt_label!r}  (source: {gt_source})")
        print(f"current bleed_label:     {row['bleed_label'] if row else '<row missing?>'}")
        print(f"current notes:           {row['notes'] if row else ''}")
    notes_box3.value = row["notes"] if row else ""
    review_state["pending_label"] = row["bleed_label"] if row else None


def set_pending(label):
    def _handler(_):
        review_state["pending_label"] = label
        with status_out3:
            clear_output(wait=True)
            print(f"Pending (not saved yet): {label} -- click 'Save correction' to write it.")
    return _handler


def on_save(_):
    if not review_paths:
        return
    rel_path = review_paths[review_state["pos"]]
    fieldnames, rows = read_all_labeled_rows()  # fresh read, right before writing
    updated = False
    for r in rows:
        if r["image_path"] == rel_path:
            r["bleed_label"] = review_state["pending_label"]
            r["notes"] = notes_box3.value.strip()
            r["timestamp"] = datetime.now().isoformat()
            updated = True
            break
    if not updated:
        with status_out3:
            clear_output(wait=True)
            print(f"\u26a0\ufe0f  {rel_path} no longer found in CSV -- not saved.")
        return
    with open(OUTPUT_CSV, "w", newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    with status_out3:
        clear_output(wait=True)
        print(f"\u2713 Saved correction for {rel_path} (row overwritten in place, no new row appended)")


def on_back(_):
    if review_paths and review_state["pos"] > 0:
        review_state["pos"] -= 1
        show_current()


def on_next(_):
    if review_paths and review_state["pos"] < len(review_paths) - 1:
        review_state["pos"] += 1
        show_current()


btn_clean3.on_click(set_pending("clean_single_column"))
btn_right3.on_click(set_pending("right_column_bleed"))
btn_below3.on_click(set_pending("below_column_bleed"))
btn_other3.on_click(set_pending("other_tangled"))
btn_skip3.on_click(set_pending("skip"))
btn_save3.on_click(on_save)
btn_back3.on_click(on_back)
btn_next3.on_click(on_next)

show_current()

In [ ]:
# ---- CELL 6: Spot-check sample of right_column_bleed rows ----
#
# Cheap manual confidence check before treating tall_crop_bleed_labels.csv
# as ground truth -- pulls a small random sample from the 218
# right_column_bleed rows (87.6% of all 249) and pages through them for a
# quick "does this actually look right" pass. Not a full re-review, and
# does NOT touch any row other than whichever one you actively correct.
#
# Reuses read_all_labeled_rows() and lookup_ground_truth() already defined
# in Cell 5 (run Cell 5 first if this errors with a NameError) -- same
# ground-truth source (val.txt, falling back to zero_shot_rotation_test's
# results.csv for paths val.txt no longer carries) and the same
# in-place-overwrite correction mechanism as the review cell, not a
# separate implementation.
#
# Does NOT touch Cell 3, does NOT re-sample or rewrite the CSV structure on
# its own -- only reads it, samples from the right_column_bleed subset, and
# (only if you click a category button + "Save correction") overwrites
# that one row in place, exactly like Cell 5.

import random

SPOTCHECK_SAMPLE_SIZE = 18  # within the requested 18-20 range

_fieldnames_sc, _rows_sc = read_all_labeled_rows()
right_bleed_rows = [r for r in _rows_sc if r["bleed_label"] == "right_column_bleed"]
print(f"right_column_bleed rows available: {len(right_bleed_rows)}")

spotcheck_paths = [r["image_path"] for r in
                    random.sample(right_bleed_rows, min(SPOTCHECK_SAMPLE_SIZE, len(right_bleed_rows)))]
print(f"Sampled {len(spotcheck_paths)} rows for spot-check (no fixed seed -- rerun this cell for a fresh sample):")
for p in spotcheck_paths:
    print(" ", p)

spotcheck_state = {"pos": 0, "pending_label": None}

image_out4 = widgets.Output()
status_out4 = widgets.Output()
notes_box4 = widgets.Text(layout=widgets.Layout(width="300px"))
btn_clean4 = widgets.Button(description="Clean single-column", button_style="success")
btn_right4 = widgets.Button(description="Right-column bleed", button_style="warning")
btn_below4 = widgets.Button(description="Below-column bleed", button_style="warning")
btn_other4 = widgets.Button(description="Other/tangled", button_style="danger")
btn_skip4 = widgets.Button(description="Skip")
btn_save4 = widgets.Button(description="Save correction", button_style="info")
btn_back4 = widgets.Button(description="< Back")
btn_next4 = widgets.Button(description="Next >")

controls4a = widgets.HBox([btn_back4, btn_next4])
controls4b = widgets.HBox([btn_clean4, btn_right4, btn_below4, btn_other4, btn_skip4])
controls4c = widgets.HBox([notes_box4, btn_save4])
display(image_out4, controls4a, controls4b, controls4c, status_out4)


def spotcheck_current_row():
    _, rows = read_all_labeled_rows()
    by_path = {r["image_path"]: r for r in rows}
    rel_path = spotcheck_paths[spotcheck_state["pos"]]
    return by_path.get(rel_path)


def spotcheck_show():
    if not spotcheck_paths:
        with image_out4:
            clear_output(wait=True)
            print("No right_column_bleed rows to sample.")
        return
    rel_path = spotcheck_paths[spotcheck_state["pos"]]
    row = spotcheck_current_row()
    with image_out4:
        clear_output(wait=True)
        img = Image.open(os.path.join(DATASET_DIR, rel_path)).convert("RGB")
        display(img)
        print(f"{rel_path}  \u2014  {spotcheck_state['pos'] + 1}/{len(spotcheck_paths)}  (spot-check sample)")
        gt_label, gt_source = lookup_ground_truth(rel_path)
        print(f"ground_truth: {gt_label!r}  (source: {gt_source})")
        print(f"current bleed_label: {row['bleed_label'] if row else '<row missing?>'}")
    notes_box4.value = row["notes"] if row else ""
    spotcheck_state["pending_label"] = row["bleed_label"] if row else None


def spotcheck_set_pending(label):
    def _handler(_):
        spotcheck_state["pending_label"] = label
        with status_out4:
            clear_output(wait=True)
            print(f"Pending (not saved yet): {label} -- click 'Save correction' to write it.")
    return _handler


def spotcheck_on_save(_):
    if not spotcheck_paths:
        return
    rel_path = spotcheck_paths[spotcheck_state["pos"]]
    fieldnames, rows = read_all_labeled_rows()  # fresh read, right before writing
    updated = False
    for r in rows:
        if r["image_path"] == rel_path:
            r["bleed_label"] = spotcheck_state["pending_label"]
            r["notes"] = notes_box4.value.strip()
            r["timestamp"] = datetime.now().isoformat()
            updated = True
            break
    if not updated:
        with status_out4:
            clear_output(wait=True)
            print(f"\u26a0\ufe0f  {rel_path} no longer found in CSV -- not saved.")
        return
    with open(OUTPUT_CSV, "w", newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    with status_out4:
        clear_output(wait=True)
        print(f"\u2713 Saved correction for {rel_path} (row overwritten in place, no new row appended)")


def spotcheck_on_back(_):
    if spotcheck_paths and spotcheck_state["pos"] > 0:
        spotcheck_state["pos"] -= 1
        spotcheck_show()


def spotcheck_on_next(_):
    if spotcheck_paths and spotcheck_state["pos"] < len(spotcheck_paths) - 1:
        spotcheck_state["pos"] += 1
        spotcheck_show()


btn_clean4.on_click(spotcheck_set_pending("clean_single_column"))
btn_right4.on_click(spotcheck_set_pending("right_column_bleed"))
btn_below4.on_click(spotcheck_set_pending("below_column_bleed"))
btn_other4.on_click(spotcheck_set_pending("other_tangled"))
btn_skip4.on_click(spotcheck_set_pending("skip"))
btn_save4.on_click(spotcheck_on_save)
btn_back4.on_click(spotcheck_on_back)
btn_next4.on_click(spotcheck_on_next)

spotcheck_show()

In [ ]:
# ---- CELL 7: Gap-detection heuristic -- visualization only, no correction ----
#
# Pages through the 218 right_column_bleed crops with the same 3-method
# overlay used in the saved contact sheets
# (data/paddle_finetune/gap_detection_viz/contact_sheets/), for a closer
# one-by-one look than a small grid thumbnail allows.
#
# Detection + visualization only: does NOT crop, save cropped images, or
# touch OCR/inference code. Does NOT modify tall_crop_bleed_labels.csv or
# gap_detection_results.csv -- only reads the former (via
# read_all_labeled_rows(), already defined in Cell 5) to build the queue,
# and calls into scripts/paddle_finetune/gap_detection.py fresh per crop
# rather than reading the precomputed results.csv, so what you see here is
# always live/recomputed, not a cached copy.
#
# Ground truth reuses lookup_ground_truth() from Cell 5 (same val.txt +
# rotation-results fallback). No correction buttons here on purpose -- this
# cell is about judging the HEURISTIC's line placement, not re-labeling
# bleed category (use Cell 5/6 for that).

import sys
sys.path.insert(0, os.path.join(REPO_ROOT, "scripts/paddle_finetune"))
from gap_detection import detect_all_methods
from PIL import ImageDraw

GAPVIZ_COLORS = {
    "column_variance": (255, 60, 60),
    "column_bg_deviation": (60, 200, 60),
    "column_edge_density": (60, 140, 255),
}

_fieldnames_gv, _rows_gv = read_all_labeled_rows()
gapviz_paths = [r["image_path"] for r in _rows_gv if r["bleed_label"] == "right_column_bleed"]
print(f"right_column_bleed crops queued for gap-detection review: {len(gapviz_paths)}")

gapviz_state = {"pos": 0}

image_out5 = widgets.Output()
btn_back5 = widgets.Button(description="< Back")
btn_next5 = widgets.Button(description="Next >")
controls5 = widgets.HBox([btn_back5, btn_next5])
display(image_out5, controls5)


def gapviz_show():
    if not gapviz_paths:
        with image_out5:
            clear_output(wait=True)
            print("No right_column_bleed rows found.")
        return
    rel_path = gapviz_paths[gapviz_state["pos"]]
    full_path = os.path.join(DATASET_DIR, rel_path)
    results = detect_all_methods(full_path)
    img = Image.open(full_path).convert("RGB")
    W, H = img.size
    draw = ImageDraw.Draw(img)
    for method, color in GAPVIZ_COLORS.items():
        res = results[method]
        if res["gap_x"] is not None:
            x = int(round(res["gap_x"]))
            draw.line([(x, 0), (x, H)], fill=color, width=max(1, W // 150))
    with image_out5:
        clear_output(wait=True)
        display(img)
        print(f"{rel_path}  \u2014  {gapviz_state['pos'] + 1}/{len(gapviz_paths)}")
        gt_label, gt_source = lookup_ground_truth(rel_path)
        print(f"ground_truth: {gt_label!r}  (source: {gt_source})")
        print("red=column_variance   green=column_bg_deviation   blue=column_edge_density")
        for method in GAPVIZ_COLORS:
            res = results[method]
            print(f"  {method}: gap_x={res['gap_x']}, gap_frac={None if res['gap_x'] is None else round(res['gap_x']/W, 3)}, "
                  f"run_width={res['run_width']}, ambiguous={res['ambiguous']}")


def gapviz_on_back(_):
    if gapviz_paths and gapviz_state["pos"] > 0:
        gapviz_state["pos"] -= 1
        gapviz_show()


def gapviz_on_next(_):
    if gapviz_paths and gapviz_state["pos"] < len(gapviz_paths) - 1:
        gapviz_state["pos"] += 1
        gapviz_show()


btn_back5.on_click(gapviz_on_back)
btn_next5.on_click(gapviz_on_next)

gapviz_show()